08_fairness_and_export

In [ ]:
import warnings, sys, shutil, joblib
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import sklearn
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

SEED = 42
OUT_DIR = Path("outputs")

fair1 = pd.read_csv(OUT_DIR / "stage1_subgroups.csv")
fair1["calibration_gap (predicted - actual rate)"] = (fair1["predicted_rate"] - fair1["actual_rate"]).round(3)
nonstate = fair1[fair1["variable"] != "state"]
display(nonstate)

summ = (nonstate[nonstate["n"] >= 300].groupby("variable")["recall"]
        .agg(lowest="min", highest="max").assign(gap=lambda t: (t["highest"] - t["lowest"]).round(3)))
print("Spread of recall across groups (groups with n >= 300):"); display(summ.round(3))

plot = nonstate[nonstate["n"] >= 300].copy()
plot["label"] = plot["variable"] + ": " + plot["group"].astype(str)
plot = plot.sort_values("recall")
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(plot["label"], plot["recall"], color="#1F3A5F")
ax.set_xlabel("Recall (share of participating women the model catches)"); ax.set_title("Stage 1: recall by group")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_stage1_fairness.png", dpi=150); plt.show()

st = fair1[fair1["variable"] == "state"].sort_values("recall")
print("States with the lowest recall:"); display(st.head(8)); print("States with the highest recall:"); display(st.tail(5))

In [ ]:
VL = pd.read_pickle(OUT_DIR / "value_labels.pkl")
SRC = {"sector": "b1q3_perv1", "social_group": "b3q4_hhv1", "religion": "b3q3_hhv1", "state": "state_perv1"}

def lab(series, var):
    m = {}
    for k, v in VL.get(SRC.get(var, var), {}).items():
        try: m[str(int(float(k)))] = v
        except Exception: m[str(k)] = v
    s = series.astype(str).str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"^0+(?=\d)", "", regex=True)
    return s.map(m).fillna(s)

stage2 = pd.read_pickle(OUT_DIR / "stage2_working_women.pkl")
specs = joblib.load(OUT_DIR / "stage2_model_specs.joblib")
FEATURES, thr = specs["features"], specs["main"]["threshold"]

dd = stage2[stage2["y_gig_vs_trad"].notna()].reset_index(drop=True)
y, g = dd["y_gig_vs_trad"].astype(int), dd["HHID"]
tri, tei = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(dd, y, g))
pipe = clone(specs["main"]["pipe"]).fit(dd.loc[tri, FEATURES], y.iloc[tri])
te = dd.iloc[tei].reset_index(drop=True)
yte = y.iloc[tei].to_numpy()
p = pipe.predict_proba(te[FEATURES])[:, 1]
print("Stage 2 test set:", len(te), "| gig-proxy share %.3f | threshold %.2f" % (yte.mean(), thr))

rows = []
for col in ["sector", "social_group", "religion"]:
    labels = lab(te[col], col).to_numpy()
    for grp in sorted(set(labels)):
        m = labels == grp
        if m.sum() < 100 or yte[m].sum() < 15 or yte[m].sum() == m.sum():
            continue
        pred = p[m] >= thr
        rows.append({"variable": col, "group": grp, "n": int(m.sum()), "gig_positives": int(yte[m].sum()),
                     "actual_rate": yte[m].mean(), "predicted_rate": pred.mean(),
                     "recall": recall_score(yte[m], pred), "precision": precision_score(yte[m], pred, zero_division=0),
                     "roc_auc": roc_auc_score(yte[m], p[m])})
fair2 = pd.DataFrame(rows).round(3)
display(fair2); fair2.to_csv(OUT_DIR / "stage2_subgroups.csv", index=False)

In [ ]:
s2 = (fair2[fair2["n"] >= 300].groupby("variable")["recall"]
      .agg(lowest="min", highest="max").assign(gap=lambda t: (t["highest"] - t["lowest"]).round(3)))
print("Stage 2: spread of recall across groups (groups with n >= 300):")
display(s2.round(3)); s2.round(3).to_csv(OUT_DIR / "stage2_fairness_gaps.csv")

fair2["flag_overshoot (predicted - actual rate)"] = (fair2["predicted_rate"] - fair2["actual_rate"]).round(3)
display(fair2[["variable", "group", "actual_rate", "predicted_rate", "flag_overshoot (predicted - actual rate)"]])

In [ ]:
def rf_row(path):
    t = pd.read_csv(path)
    return t[t["model"] == "Random forest"].iloc[0]

key = {}
try:
    r = rf_row(OUT_DIR / "stage1_test_results.csv")
    key["Stage 1 (participation), random forest, test"] = {k: r[k] for k in ["roc_auc", "pr_auc", "precision", "recall", "f1"]}
except Exception as e: print("Stage 1 results missing:", e)
try:
    r = rf_row(OUT_DIR / "stage2_main_test.csv")
    key["Stage 2 (gig-proxy vs traditional), random forest, test"] = {k: r[k] for k in ["roc_auc", "pr_auc", "precision", "recall", "f1"]}
except Exception as e: print("Stage 2 results missing:", e)
key_tbl = pd.DataFrame(key).T.round(3); display(key_tbl); key_tbl.to_csv(OUT_DIR / "key_numbers_models.csv")

try:
    lat = pd.read_csv(OUT_DIR / "latent_headline.csv", index_col=0)
    print("\nLatent gig-like women (check the weight scale before quoting weighted millions):"); display(lat.round(3))
except Exception as e: print("latent_headline.csv missing:", e)

try:
    sg = pd.read_csv(OUT_DIR / "state_residuals_main.csv", index_col=0)
    print("\nLargest negative state gaps:"); display(sg[["n", "actual", "predicted", "gap"]].head(5))
    print("Largest positive state gaps:"); display(sg[["n", "actual", "predicted", "gap"]].tail(5))
except Exception as e: print("state residuals missing:", e)

In [ ]:
ASSETS = OUT_DIR / "report_assets"
ASSETS.mkdir(exist_ok=True)
copied = []
for f in sorted(OUT_DIR.glob("*.csv")) + sorted(OUT_DIR.glob("*.png")):
    shutil.copy(f, ASSETS / f.name); copied.append(f.name)
print(len(copied), "files copied to", ASSETS)
for n in copied: print(" -", n)

readme = f"""Door or Trap? Women, Work and the Gig Economy in India: PLFS 2023-24
Run order (Restart kernel and Run All in each section):
 01_data_preparation  -> saves stage1/stage2/outside/seeking tables, feature lists, value labels
 02_eda               -> exploratory tables and figures
 03_stage1_models     -> participation models, threshold, subgroup performance
 04_stage2_models     -> gig-proxy vs traditional models, saves model specs
 05_latent_gig_eligible -> calibrated scoring of women outside the labour force
 06_state_residual_map -> state gaps from models without state
 07_door_or_trap      -> earnings, welfare, social security, contracts
 08_fairness_and_export -> subgroup checks and file collection
Input: PERV MER COPY - Copy.sav (set DATA_PATH in 01). All other files are written to outputs/.
Notes: economy_type was built in SPSS (syntax and NCO code lists are in the report appendix).
Weights: PLFS multiplier/100 if NSS == NSC, else /200, then x0.25 (four pooled quarters; confirm in the manual).
Environment: Python {sys.version.split()[0]}, scikit-learn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}
"""
(ASSETS / "README_run_order.txt").write_text(readme)
print(readme)